## 1.0 Data Acquisition and Loading

### 1.1. Load libraries and datasets

In [ ]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

#### Display Train data

In [ ]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_len = len(train_data)
train_data.head()

#### Display Test data

In [ ]:
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
test_data.head()

#### Data Dictionary

| Variable | Definition | Key |
| :--- | :--- | :--- |
| **Survived** | Survival | 0 = No, 1 = Yes |
| **Pclass** | Ticket class | 1 = 1st, 2 = 2nd, 3 = 3rd |
| **Sex** | Sex | |
| **Age** | Age in years | |
| **SibSp** | # of siblings / spouses aboard | |
| **Parch** | # of parents / children aboard | |
| **Ticket** | Ticket number | |
| **Fare** | Passenger fare | |
| **Cabin** | Cabin number | |
| **Embarked** | Port of Embarkation | C = Cherbourg, Q = Queenstown, S = Southampton |

### 1.2 Explore some basic pattern

In [ ]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

In [ ]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})

total = len(train_data)

survived = (train_data.Survived == 1).sum()

median_age = train_data['Age'].median()
minors = (train_data['Age'].fillna(median_age) < 18).sum()

survived_fraction = survived / total

minors_fraction = minors / total 

## 2.0 Feature Engineering and Imputation

### 2.1 Imputation Statistics and Initial Feature Creation

This section calculates key statistical values from the training set, which are necessary for consistent imputation on both data sets. It also extracts the Deck letter from the Cabin column, e.g., C123 → C.

In [ ]:
train_data['Deck'] = train_data['Cabin'].str[0]
test_data['Deck'] = test_data['Cabin'].str[0]

deck_by_pclass = train_data[train_data['Deck'] != 'Z'].groupby('Pclass')['Deck'].agg(lambda x: x.mode()[0])

median_fare_train = train_data['Fare'].median()

### 2.2 Deck and Fare Imputation

This code imputes missing Deck values ('Z') with the most frequent Deck of their respective Passenger Class. It also fills the missing Fare value in the test set.

In [ ]:
train_data['Cabin'] = train_data['Cabin'].fillna('Z')
test_data['Cabin'] = test_data['Cabin'].fillna('Z')

def impute_deck(df, deck_mode_map):
    for pclass in df['Pclass'].unique():
        if pclass in deck_mode_map:
            mode_deck = deck_mode_map[pclass]
            df.loc[(df['Deck'] == 'Z') & (df['Pclass'] == pclass), 'Deck'] = mode_deck
    return df

train_data = impute_deck(train_data, deck_by_pclass)
test_data = impute_deck(test_data, deck_by_pclass)

test_data['Fare'] = test_data['Fare'].fillna(median_fare_train)

## 3.0 Final Data Preparation

### 3.1 Concatenation and One-Hot Encoding

The training and test data are combined to ensure consistent One-Hot Encoding across both sets. The categorical Deck feature is converted into binary columns.

In [ ]:
combined_data = pd.concat([train_data.drop('Survived', axis=1), test_data], ignore_index=True)

combined_data = pd.get_dummies(combined_data, columns=['Deck'], prefix='Deck')

combined_data = pd.get_dummies(combined_data, columns=['Sex', 'Embarked'], prefix=['Sex', 'Embarked'])

### 3.2: Title Feature Engineering

The title (e.g., Mr., Miss, Dr.) is extracted from the name because it has a strong correlation with the survival rate (status and gender). Rare titles are grouped together to create robust categories for the model.

In [ ]:
combined_data['Title'] = combined_data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

rare_titles = ['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
combined_data['Title'] = combined_data['Title'].replace(rare_titles, 'Rare')
combined_data['Title'] = combined_data['Title'].replace(['Mlle', 'Ms'], 'Miss')
combined_data['Title'] = combined_data['Title'].replace('Mme', 'Mrs')

combined_data = pd.get_dummies(combined_data, columns=['Title'], prefix='Title')

title_cols = [col for col in combined_data.columns if col.startswith('Title_')]

### 3.2.1 Splitting the Data and removing redundant Features

The combined data set is split back into processed training and test data frames. The target variable 'Survived' is re-added to the training set. Original columns that are no longer needed (Strings or already encoded) are dropped. This improves code hygiene and prevents the ML model from crashing.

In [ ]:
train_data_processed = combined_data.iloc[:train_len, :].copy()
test_data_processed = combined_data.iloc[train_len:, :].copy()

train_data_processed['Survived'] = train_data['Survived']

cols_to_drop = ['Cabin', 'Ticket', 'Name'] 

train_data_processed.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')
test_data_processed.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')

### 3.2.2 Final Feature Correction

Duplicate columns and any remaining NaN values are corrected. All Deck features are explicitly cast to boolean type for model compatibility.

In [ ]:
cols_to_drop = ['Cabin', 'Ticket', 'Name'] 
train_data_processed.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')
test_data_processed.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')

test_data_processed = test_data_processed.loc[:,~test_data_processed.columns.duplicated(keep='last')]
train_data_processed = train_data_processed.loc[:,~train_data_processed.columns.duplicated(keep='last')]

test_data_processed = test_data_processed.fillna(False)
train_data_processed = train_data_processed.fillna(False) 

deck_cols = ['Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T']

for col in deck_cols:
    if col in test_data_processed.columns:
        test_data_processed[col] = test_data_processed[col].astype(bool)
    if col in train_data_processed.columns:
        train_data_processed[col] = train_data_processed[col].astype(bool)

relevant_cols = [
    'PassengerId', 'Deck_A', 'Deck_B', 'Deck_C', 
    'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T'
]

## 4.0 Modeling and Prediction

This section selects the final features, trains a Random Forest Classifier on the training data, and generates the final predictions.

### 4.1 Feature Selection and Model Training

The final feature set (X) is selected, excluding IDs and target variables. The Random Forest model is initialized and fitted to the training data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

X_train = train_data_processed.drop(['PassengerId', 'Survived'], axis=1)
y_train = train_data_processed['Survived']
X_test = test_data_processed.drop('PassengerId', axis=1)

X_train['Age'] = pd.to_numeric(X_train['Age'], errors='coerce')
X_test['Age'] = pd.to_numeric(X_test['Age'], errors='coerce')

median_age = X_train['Age'].median()
median_fare = X_train['Fare'].median()

X_train['Age'] = X_train['Age'].fillna(median_age)
X_test['Age'] = X_test['Age'].fillna(median_age)

X_test['Fare'] = X_test['Fare'].fillna(median_fare)

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)

model.fit(X_train, y_train)

### 4.2 Prediction and Submission

The trained model is used to predict the survival on the test data. The results are formatted into the required Kaggle submission CSV file.

In [ ]:
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test_data_processed['PassengerId'],
    'Survived': predictions
})

submission['Survived'] = submission['Survived'].astype(int)

submission.to_csv('submission.csv', index=False) 

print("\nSubmission file saved as 'submission.csv'")
print("\nFirst 5 rows of the generated submission file:\n")
print(submission.head())

print('\nYour submission file was successfully created.')